In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
train_trans = pd.read_csv("D:/project/data/train_transaction.csv")
train_id = pd.read_csv("D:/project/data/train_identity.csv")
test_trans = pd.read_csv("D:/project/data/test_transaction.csv")
test_id = pd.read_csv("D:/project/data/test_identity.csv")

#merge to 1
test_df = test_trans.merge(test_id,on="TransactionID",how="left")
train_df=train_trans.merge(train_id,on="TransactionID",how="left")




# Dataset Cleaning and Preprocessing

This section describes the data preprocessing pipeline applied to the IEEE-CIS Fraud Detection dataset, including the following steps:

- Remove the label column **`isFraud`** from the feature set.
- Identify and handle missing values.
- Encode categorical features into numerical representations.
- Ensure that training and testing datasets share the same feature columns.
- Split the dataset into training and validation sets based on transaction time.


In [ ]:
y = train_df["isFraud"].copy()
X = train_df.drop(columns=["isFraud"]).copy()
X_test = test_df.copy()

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keep

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\nTrain fraud ratio:")
print(y_train.value_counts(normalize=True))

print("\nVal fraud ratio:")
print(y_val.value_counts(normalize=True))

In [ ]:
# đảm bảo các bộ dữ liệu có cùng cột
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=np.nan)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=np.nan)

# sau khi align với train, align lại val theo train cho chắc
X_val = X_val.reindex(columns=X_train.columns, fill_value=np.nan)
X_test = X_test.reindex(columns=X_train.columns, fill_value=np.nan)

print("Aligned shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

Saved train_processed.csv & test_processed.csv


In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of categorical columns:", len(cat_cols))
print("Number of numerical columns  :", len(num_cols))

In [ ]:
# Numerical: median từ train
for col in num_cols:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_val[col]   = X_val[col].fillna(median_value)
    X_test[col]  = X_test[col].fillna(median_value)

# Categorical: điền "missing"
for col in cat_cols:
    X_train[col] = X_train[col].fillna("missing")
    X_val[col]   = X_val[col].fillna("missing")
    X_test[col]  = X_test[col].fillna("missing")

In [ ]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
X_val[cat_cols]   = encoder.transform(X_val[cat_cols].astype(str))
X_test[cat_cols]  = encoder.transform(X_test[cat_cols].astype(str))

In [ ]:
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_val   = X_val.apply(pd.to_numeric, errors="coerce")
X_test  = X_test.apply(pd.to_numeric, errors="coerce")

# Nếu có NaN phát sinh sau encode/coerce thì fill lại 0
X_train = X_train.fillna(0)
X_val   = X_val.fillna(0)
X_test  = X_test.fillna(0)

In [ ]:
print("Before balancing:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

In [ ]:
smote = SMOTE(
    sampling_strategy=0.5,   # fraud = 50% normal
    random_state=42,
    k_neighbors=5
)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print(pd.Series(y_train_bal).value_counts(normalize=True))

In [ ]:
train_processed = X_train.copy()
train_processed["isFraud"] = y_train.values

val_processed = X_val.copy()
val_processed["isFraud"] = y_val.values

train_balanced_processed = pd.DataFrame(X_train_bal, columns=X_train.columns)
train_balanced_processed["isFraud"] = y_train_bal

test_processed = X_test.copy()

In [ ]:
train_processed.to_csv("train_processed.csv", index=False)
val_processed.to_csv("val_processed.csv", index=False)
train_balanced_processed.to_csv("train_balanced_processed.csv", index=False)
test_processed.to_csv("test_processed.csv", index=False)

print("Saved:")
print("- train_processed.csv")
print("- val_processed.csv")
print("- train_balanced_processed.csv")
print("- test_processed.csv")